# 06. RTL(Verilog) post-training: CodeV-R1 따라 하기

5단계에서 SFT, DPO, GRPO를 "고양이 비서" 말투로 연습했습니다. 이번에는 같은 도구를 **진짜 도메인 문제**에 써 봅니다.
따라갈 대상은 오픈소스 프로젝트 **[CodeV-R1](https://github.com/IPRC-DIP/CodeV-R1)** 입니다.
자연어 스펙을 받아 **Verilog RTL 코드를 쓰는 7B 추론 모델**을 만든 프로젝트로, 코드, 데이터, 모델이 모두 공개되어 있습니다.

> 논문: *CodeV-R1: Reasoning-Enhanced Verilog Generation* (arXiv 2505.24183)
> 모델: `zhuyaoyu/CodeV-R1-Distill-Qwen-7B`(SFT까지), `zhuyaoyu/CodeV-R1-Qwen-7B`(RL까지) · 데이터: `zhuyaoyu/CodeV-R1-dataset`

CodeV-R1의 레시피는 한 줄로 요약됩니다.

> **큰 추론 모델(DeepSeek-R1)의 풀이를 SFT로 증류(distillation)하고, 그다음 "시뮬레이터가 채점해 주는 보상"으로 RL(DAPO)을 한다.**

Verilog는 RL에 아주 잘 맞는 분야입니다. 모델이 쓴 코드를 **시뮬레이터(Icarus Verilog)로 돌려서 정답 회로와 똑같이 동작하는지** 자동으로 채점할 수 있기 때문입니다.
5단계의 GRPO에서는 "🐾로 끝나는가" 같은 규칙으로 보상을 줬다면, 이번에는 **회로가 맞게 동작하는가**가 보상입니다.

**이 노트북에서 배우는 것**

0. CodeV-R1 전체 파이프라인과 저장소 구조
1. CodeV-R1의 데이터 형식 (SFT용 JSONL, RL용 parquet의 `ground_truth`)
2. **보상 함수의 핵심: 테스트벤치 자동 생성 + 등가성 검사** (CodeV-R1 코드를 단순화해 직접 구현)
3. 데이터 합성: round-trip(코드 → 스펙 → 코드)과 등가성 필터
4. Stage 1: 증류 SFT (LLaMA-Factory 설정을 TRL로 옮기기)
5. RL 데이터 고르기: 난이도 필터
6. Stage 2: RL (DAPO와 CodeV-R1의 adaptive DAPO, verl 설정을 TRL로 옮기기)
7. 베이스, SFT, RL 모델을 pass@k로 비교하기
8. 실제 규모로 돌리는 방법

**실행 환경**

| 환경 | 예상 시간 (전체) | 필요 메모리 |
|---|---|---|
| RTX 3060 12GB | 약 30~50분 (생성이 대부분) | VRAM 약 6~9GB |
| Apple Silicon Mac (M1~M4, `mps`) | 측정 안 함 (RTX 3060보다 느릴 것으로 예상) | 통합 메모리 16GB 이상 권장 |
| CPU만 있는 경우 | 1~2시간 (샘플 수, 스텝 수를 줄인 설정) | RAM 8GB 이상 |

> ⚠️ 이 노트북은 **Icarus Verilog(`iverilog`)** 가 설치되어 있어야 합니다. 바로 아래 "준비" 절에서 설치 방법을 안내합니다.
> 처음 실행할 때 Hugging Face Hub에서 모델(약 1GB)을 내려받으므로 인터넷 연결이 필요합니다.

## 0. 큰 그림: CodeV-R1은 어떻게 만들어졌나

```
 ┌──────────────────────────── 데이터 만들기 ────────────────────────────┐
 │ GitHub의 진짜 Verilog 코드                                             │
 │      │ DeepSeek-V3: "이 코드의 스펙을 자연어로 써 줘"   (code → NL)     │
 │      ▼                                                                │
 │ (스펙, 원본 코드) 약 150K                                              │
 │      │ DeepSeek-R1: 스펙만 보고 생각(<think>) + 코드 작성 (NL → code)   │
 │      ▼                                                                │
 │ 원본 코드 vs R1 코드 등가성 검사 (자동 생성 테스트벤치 + iverilog)       │
 │ + 쉬운 문제 제거, 벤치마크와 겹치는 문제 제거                           │
 └──────┬─────────────────────────────────────────────┬──────────────────┘
        ▼                                             ▼
  SFT 데이터 약 87K                               RL 데이터 3.1K (어려운 문제만)
        │                                             │
        ▼                                             ▼
 ① 증류 SFT (LLaMA-Factory)                   ② RL: adaptive DAPO (verl)
 Qwen2.5-Coder-7B-Instruct → CodeV-R1-Distill  → CodeV-R1-7B
 "R1처럼 생각하고 쓰는 법"을 따라 배움            보상 = 형식 OK && 원본 코드와 등가 ? 1 : 0
```

5단계와 나란히 놓으면 이렇습니다.

| | 5단계 (냥이) | CodeV-R1 원본 | 이번 실습 (축소판) |
|---|---|---|---|
| 출발 모델 | Qwen2.5-0.5B (base) | Qwen2.5-Coder-**7B**-Instruct | Qwen2.5-Coder-**0.5B**-Instruct |
| SFT 데이터 | 사람이 쓴 냥이 답변 | R1이 쓴 풀이 중 **검증 통과한 것** 87K | 교사 응답 중 검증 통과한 것 약 20개 |
| SFT 방식 | TRL + LoRA | LLaMA-Factory, **Full**, 6 epoch, 16K 길이 | TRL + LoRA |
| RL 보상 | 말투 규칙 점수 | **시뮬레이션 등가성 (0 또는 1)** | 같은 방식 (단순화한 테스트벤치) |
| RL 알고리즘 | GRPO | **DAPO** (+ adaptive 샘플링), verl | TRL `GRPOTrainer`의 `loss_type="dapo"` |
| RL 데이터 | 질문 목록 | 3.1K | 12개 |
| 계산량 | 수 분 | SFT A100×8 78시간 + RL A100×16 127시간 (약 2,656 GPU시간) | RTX 3060 수십 분 |

> 💡 숫자는 작지만 **구조는 원본과 같습니다**. 데이터 형식, 보상 함수, 학습 설정 이름을 원본 코드와 하나씩 맞춰 가며 진행하므로,
> 이 노트북을 이해하면 원본 저장소의 스크립트를 그대로 읽을 수 있습니다.

### CodeV-R1 저장소 구조와 이 노트북의 대응

| 원본 경로 | 하는 일 | 이 노트북 |
|---|---|---|
| `sft/codev_r1_distill.yaml` | LLaMA-Factory SFT 설정 | 4절 |
| `verl/recipe/dapo/run_dapo_codev_7b_3.1k.sh` | verl RL 실행 스크립트 (DAPO 설정) | 6절 |
| `verl/verl/utils/reward_score/codev.py` | 보상 함수 `compute_score` (형식 + 등가성) | 2-5절 |
| `verl/.../codev_eval_toolkit/verify.py` | 테스트벤치 자동 생성, iverilog 실행 | 2-3, 2-4절 |
| `verl/.../codev_eval_toolkit/eval_codev.py` | 코드 추출, 타임아웃 처리 | 2-1, 2-4절 |
| `test/` | VerilogEval, RTLLM 벤치마크 평가 | 7절 (pass@k) |

## 준비. 환경 설정

### Icarus Verilog 설치

보상 함수가 Verilog 시뮬레이터를 직접 실행하므로 `iverilog`(컴파일러)와 `vvp`(시뮬레이터 실행기)가 필요합니다. CodeV-R1도 같은 도구를 씁니다 (`conda create -n codev-r1 ... conda-forge::iverilog`).

| OS | 설치 명령 |
|---|---|
| Ubuntu / WSL2 | `sudo apt install iverilog` |
| macOS (Apple Silicon 포함) | `brew install icarus-verilog` |
| conda (모든 OS) | `conda install -c conda-forge iverilog` |
| Windows (네이티브) | https://bleyer.org/icarus/ 에서 설치 파일을 받고, 설치 폴더의 `bin`을 PATH에 추가 |

> 💡 Windows라면 **WSL2 + Ubuntu**를 권장합니다. GPU(CUDA)도 WSL2에서 그대로 쓸 수 있고, 원본 CodeV-R1 스크립트도 리눅스 기준입니다.
>
> 🍎 Apple Silicon Mac(M1~M4)은 `brew install icarus-verilog`로 설치하면 되고, 아래 셀이 자동으로 Mac GPU(`mps`)를 고릅니다.
> 원본 CodeV-R1의 학습 스크립트(verl, vLLM, flash-attn, DeepSpeed)는 CUDA 전용이라 Mac에서는 돌릴 수 없고, 이 노트북의 축소판만 가능합니다.

설치 후 아래 셀로 확인합니다.

In [ ]:
import shutil
import subprocess

for tool in ["iverilog", "vvp"]:
    print(f"{tool:9}:", shutil.which(tool))
assert shutil.which("iverilog") and shutil.which("vvp"), "iverilog/vvp를 찾지 못했습니다. 위 표를 보고 설치한 뒤 Jupyter를 다시 시작하세요."
print(subprocess.run(["iverilog", "-V"], capture_output=True, text=True).stdout.splitlines()[0])

### 라이브러리와 실행 설정

5단계와 같은 TRL을 씁니다 (`pip install -r requirements.txt`).

In [ ]:
import gc
import json
import math
import os
import random
import re
import tempfile
import time
from concurrent.futures import ThreadPoolExecutor
from itertools import combinations
from pathlib import Path

import torch
import transformers
import datasets
import peft
import trl

print("PyTorch      :", torch.__version__)
print("transformers :", transformers.__version__)
print("peft         :", peft.__version__)
print("trl          :", trl.__version__)

if torch.cuda.is_available():
    device = "cuda"                      # NVIDIA GPU
elif torch.backends.mps.is_available():
    device = "mps"                       # Apple Silicon (M1~M4) GPU
else:
    device = "cpu"
print("사용할 장치  :", device)
if device == "cuda":
    props = torch.cuda.get_device_properties(0)
    print("GPU 이름     :", props.name)
    print(f"VRAM         : {props.total_memory / 1024**3:.1f} GB")

transformers.set_seed(42)
random.seed(42)

아래 `config`가 이번 실습의 설정입니다. CPU에서는 생성 길이, 샘플 수, 스텝 수를 줄였습니다.

- **`eval_samples`:** 평가 때 문제 하나당 몇 개의 답을 뽑아 pass@1을 낼지. 원본 논문은 **20개**를 뽑습니다.
- **`gen_max_new_tokens`:** 생성 최대 길이. 원본은 생각 과정이 길어서 **16,384 토큰**까지 씁니다. 우리 교사 응답은 짧아서 수백 토큰이면 됩니다.
- **`grpo_num_generations`:** 프롬프트 하나당 RL 답안 수(그룹 크기 G). 원본은 **16**.

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-Coder-0.5B-Instruct"   # CodeV-R1은 Qwen2.5-Coder-7B-Instruct에서 출발

if device == "cuda":
    use_bf16 = torch.cuda.is_bf16_supported()
    config = dict(
        bf16=use_bf16, fp16=not use_bf16,
        gen_max_new_tokens=512, eval_samples=4, eval_temperature=0.6, gen_batch=8,
        sft_epochs=4, sft_lr=2e-4, sft_batch_size=4, sft_max_length=1024,
        filter_samples=4,
        grpo_steps=40, grpo_lr=2e-5, grpo_num_generations=8, grpo_batch_size=8, grpo_grad_accum=2,
        grpo_max_completion=512,
    )
elif device == "mps":
    # Apple Silicon: 혼합 정밀도(bf16/fp16) 학습은 MPS에서 불안정할 수 있어 fp32로 둔다.
    # 0.5B 모델은 fp32여도 약 2GB라 통합 메모리 16GB면 충분하다. 스텝 수는 GPU 설정의 절반.
    config = dict(
        bf16=False, fp16=False,
        gen_max_new_tokens=512, eval_samples=4, eval_temperature=0.6, gen_batch=8,
        sft_epochs=4, sft_lr=2e-4, sft_batch_size=4, sft_max_length=1024,
        filter_samples=4,
        grpo_steps=20, grpo_lr=2e-5, grpo_num_generations=8, grpo_batch_size=8, grpo_grad_accum=1,
        grpo_max_completion=512,
    )
else:
    config = dict(
        bf16=False, fp16=False,
        gen_max_new_tokens=320, eval_samples=2, eval_temperature=0.6, gen_batch=4,
        sft_epochs=3, sft_lr=3e-4, sft_batch_size=2, sft_max_length=768,
        filter_samples=2,
        grpo_steps=6, grpo_lr=3e-5, grpo_num_generations=4, grpo_batch_size=4, grpo_grad_accum=1,
        grpo_max_completion=320,
    )
    torch.set_num_threads(os.cpu_count())

for k, v in config.items():
    print(f"{k:22}: {v}")

ROOT = Path("..") if Path("../data").exists() else Path(".")
DATA_DIR = ROOT / "data"
OUTPUT_DIR = ROOT / "outputs" / "06_codev_r1"

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import font_manager

for font in ["Malgun Gothic", "AppleGothic", "NanumGothic", "Noto Sans CJK KR"]:
    if any(f.name == font for f in font_manager.fontManager.ttflist):
        plt.rcParams["font.family"] = font
        plt.rcParams["axes.unicode_minus"] = False
        break

def free_memory():
    gc.collect()
    if device == "cuda":
        torch.cuda.empty_cache()
    elif device == "mps":
        torch.mps.empty_cache()

def peak_vram():
    if device == "cuda":
        return torch.cuda.max_memory_allocated() / 1024**3
    if device == "mps":
        return torch.mps.driver_allocated_memory() / 1024**3   # MPS는 최댓값 대신 현재 할당량
    return float("nan")

## 1. CodeV-R1의 데이터 형식

### 1-1. 원본 데이터는 어떻게 생겼나

Hugging Face의 `zhuyaoyu/CodeV-R1-dataset`에는 파일이 세 개 있습니다.

| 파일 | 쓰임 | 형식 |
|---|---|---|
| `codev_r1_sft.jsonl` (약 87K줄) | SFT | `{"system": ..., "prompt": 스펙, "response": "<think>R1의 생각</think> <answer>```verilog ...```</answer>"}` |
| `codev_r1_rl_train.parquet` (3.1K) | RL 학습 | verl 형식: `prompt`(대화 메시지), `data_source="codev"`, `reward_model.ground_truth` |
| `codev_r1_rl_val.parquet` | RL 검증 | 같음 |

SFT 파일은 LLaMA-Factory에 이렇게 등록합니다 (`sft/README.md`). `system / prompt / response` 세 칸이 곧 2·5단계에서 다룬 **(시스템 프롬프트, 질문, 답)** 입니다.
```json
"codev_r1_sft": {
  "file_name": "your/path/to/codev_r1_sft.jsonl",
  "columns": {"system": "system", "prompt": "prompt", "response": "response"}
}
```

RL 데이터에서 가장 중요한 것은 `ground_truth`입니다. **정답 답안이 아니라 "채점에 필요한 정보"** 가 들어 있습니다. `codev.py`는 이것을 `pickle.loads`로 풀어서 씁니다.
```python
ground_truth = {
    "answer": {
        "code": "module top_module(...); ... endmodule",       # 원본(golden) 코드
        "input_port_width":  {("clk", 1), ("rst_n", 1), ("d", 8)},   # 입력 포트와 비트 폭
        "output_port_width": {("q", 8)},                            # 출력 포트와 비트 폭
        "clock_port_polarity": {("clk", 1)},                        # 클럭 이름, 1 = posedge
        "reset_port_polarity_sync": {("rst_n", 0, False)},          # 리셋 이름, 활성 값, 동기 여부
    }
}
```
> 💡 RL에는 모범 답안(풀이)이 필요 없습니다. **"정답 회로"와 "그 회로를 시뮬레이션하는 데 필요한 포트 정보"만 있으면** 모델이 어떤 코드를 쓰든 채점할 수 있습니다.

### 1-2. 이번 실습용 데이터

`data/` 폴더에 원본과 같은 구조의 작은 데이터를 준비했습니다.

| 파일 | 내용 |
|---|---|
| `rtl_problems.jsonl` | 42문제. `spec`(영어 스펙), `module_head`, `golden`(원본 코드 + 포트 정보 = `ground_truth`), `split` (`sft` 22 / `rl` 12 / `eval` 8) |
| `rtl_teacher_responses.jsonl` | `sft` 문제에 대한 "교사 모델" 응답 33개. **일부러 틀린 응답도 섞여 있습니다** (3절에서 걸러 냄) |

원본 데이터처럼 스펙은 영어로 썼습니다. 코드 모델이 영어 스펙에 더 익숙하기 때문입니다.

In [ ]:
def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

problems = read_jsonl(DATA_DIR / "rtl_problems.jsonl")
teacher_responses = read_jsonl(DATA_DIR / "rtl_teacher_responses.jsonl")
by_id = {p["id"]: p for p in problems}
splits = {s: [p for p in problems if p["split"] == s] for s in ["sft", "rl", "eval"]}

print({s: len(v) for s, v in splits.items()}, "| 교사 응답:", len(teacher_responses))
ex = by_id["counter4_en"]
print("\n[spec]\n" + ex["spec"])
print("\n[module_head]\n" + ex["module_head"])
print("\n[golden code]\n" + ex["golden"]["code"])
print("[포트 정보]")
for k in ["input_port_width", "output_port_width", "clock_port_polarity", "reset_port_polarity_sync"]:
    print(f"  {k:25}: {ex['golden'][k]}")

### 1-3. 프롬프트 형식

CodeV-R1은 DeepSeek-R1과 같은 **`<think> 생각 </think> <answer> 답 </answer>`** 형식을 씁니다.
아래 시스템 프롬프트는 원본 `test/llmkit_data/cli/prep_verilogeval.py`의 `mk_prompt_codev_rl`에서 **그대로** 가져왔습니다 (평가 때 이 프롬프트를 씀).
사용자 메시지는 VerilogEval처럼 "스펙 + `The module head should be:` + 모듈 선언부" 형태입니다.

In [ ]:
SYSTEM_PROMPT = """You are a helpful assistant. The assistant first thinks about the reasoning process in the mind and then provides the user with the answer. The reasoning process and answer are enclosed within <think> </think> and<answer> </answer> tags, respectively, i.e., <think> reasoning process here </think><answer> answer here </answer>.  Now the user asks you to write verilog code. After thinking, when you finally reach a conclusion, enclose the final verilog code in ```verilog ``` within <answer> </answer> tags. i.e., <answer> ```verilog
 module top_module(in, out, ...); ... ``` </answer>.
"""

def build_messages(problem):
    user = problem["spec"] + "\nThe module head should be:\n" + problem["module_head"] + "\n"
    return [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user}]

for m in build_messages(ex):
    print(f"--- {m['role']} ---\n{m['content']}")

## 2. 보상 함수의 핵심: 테스트벤치 기반 등가성 검사

### 왜 어려운가
"4비트 카운터"를 쓰는 방법은 무수히 많습니다. `q <= q + 1`도 되고 `q <= (q == 15) ? 0 : q + 1`도 됩니다.
그래서 **코드 문자열 비교로는 채점할 수 없고, "동작"을 비교해야 합니다.**

CodeV-R1의 방법은 이렇습니다 (`verify.py`의 `eda_tools`).

```
 원본(golden) 코드 ──┐                                  ┌─▶ golden 출력 ─┐
                     ├─ 이름 충돌 방지(_gold/_gate) ─▶ TB │                 ├─ 매 순간 비교 (===)
 모델(gate) 코드  ───┘   같은 무작위 입력을 둘 다에 넣음  └─▶ gate 출력   ─┘   하나라도 다르면 오류
                                                                          ▼
                                                     오류율 0% → 등가(정답, 보상 1) / 아니면 보상 0
```

1. **구조 분석:** 포트 이름과 비트 폭, 클럭(어느 엣지), 리셋(활성 값, 동기/비동기)을 알아냅니다. 원본은 **Yosys**로 하고, 결과를 `ground_truth`에 미리 저장해 둡니다.
2. **테스트벤치 자동 생성:** golden과 gate를 나란히 인스턴스하고, 무작위 입력 **M=100개 시퀀스 × N=1000 스텝**을 넣습니다. 순차회로는 리셋 조합별로 한 번 더 돌립니다.
3. **시뮬레이션과 비교:** iverilog로 돌려서 출력이 한 번이라도 다르면 실패입니다.

이제 이 과정을 원본 코드를 줄여 가며 직접 구현합니다. (원본은 약 500줄, 우리 버전은 약 150줄)

### 2-1. 답안에서 코드 꺼내기, 형식 검사

`eval_codev.py`의 `extract_verilog`와 `codev.py`의 `check_format`을 그대로 옮긴 것입니다.
형식 검사는 엄격합니다. **네 태그가 정확히 한 번씩, 순서대로** 나와야 합니다. 형식이 틀리면 코드가 맞아도 0점입니다.

In [ ]:
def extract_verilog(text):
    """CodeV-R1 eval_codev.extract_verilog 와 동일: 마지막 ```verilog 블록을 꺼낸다."""
    matches = re.findall(r"```verilog\s*([\s\S]*?)\s*```", text)
    return matches[-1] if matches else None


def check_format(output):
    """CodeV-R1 codev.py 의 check_format 과 동일: 네 태그가 정확히 한 번씩, 순서대로."""
    tags = ["<think>", "</think>", "<answer>", "</answer>"]
    counts = [output.count(t) for t in tags]
    pos = [output.find(t) for t in tags]
    return min(counts) == max(counts) == 1 and pos[0] < pos[1] < pos[2] < pos[3]

In [ ]:
good = "<think>easy</think> <answer>\n```verilog\nmodule top_module(); endmodule\n```\n</answer>"
bad1 = "<think> <answer></think> ```verilog\nmodule top_module(); endmodule\n```</answer>"   # 태그 순서가 틀림
bad2 = "```verilog\nmodule top_module(); endmodule\n```"                                     # 태그 없음
for name, s in [("good", good), ("bad1", bad1), ("bad2", bad2)]:
    print(f"{name}: format={check_format(s)}, code={extract_verilog(s)!r}")

### 2-2. 모듈 파싱: top 모듈 찾기, 이름 바꾸기

- **`auto_top`:** 코드에 모듈이 여러 개일 수 있습니다 (예: `full_adder`를 8번 인스턴스하는 `top_module`). 다른 모듈에게 인스턴스되지 않는 모듈 중 가장 큰 계층을 가진 것을 top으로 고릅니다. 원본은 `networkx`로 인스턴스 그래프를 만듭니다.
- **`rename_modules`:** golden과 gate가 둘 다 `top_module`이면 한 시뮬레이션에 같이 넣을 수 없으니, 모든 모듈 이름 뒤에 `_gold` / `_gate`를 붙입니다 (원본 `process_verilog`).

In [ ]:
COMMENT_RE = r"(//[^\n]*|/\*[\s\S]*?\*/)"
MODULE_DEF_RE = (r"(module\s+)([a-zA-Z_][a-zA-Z0-9_\$]*)(\s*\#\s*\([\s\S]*?\))?"
                 r"(\s*(?:\([^;]*\))?\s*;)([\s\S]*?)?(endmodule)")


def strip_comments(code):
    code = re.sub(COMMENT_RE, "", code)
    return re.sub(r"(?:\s*?\n)+", "\n", code)


def auto_top(code):
    """다른 모듈에게 인스턴스되지 않는 모듈 중 가장 큰 트리를 가진 것을 top 으로 (CodeV-R1 auto_top 단순화)."""
    defs = re.findall(MODULE_DEF_RE, strip_comments(code))
    if not defs:
        raise ValueError("module 을 찾지 못함")
    names = [d[1] for d in defs]
    children = {n: set() for n in names}
    for d in defs:
        for sub in names:
            if sub != d[1] and re.search(rf"\b{re.escape(sub)}\s+(\#\s*\([\s\S]*?\)\s*)?[a-zA-Z_]\w*\s*\(", d[4]):
                children[d[1]].add(sub)
    instantiated = set().union(*children.values())
    roots = [n for n in names if n not in instantiated]

    def size(n, seen=()):
        return 1 + sum(size(c, seen + (n,)) for c in children[n] if c not in seen)
    return max(roots, key=size)


def rename_modules(code, suffix):
    """모든 모듈 이름 뒤에 suffix 를 붙인다 (gold 와 gate 의 이름 충돌 방지). CodeV-R1 process_verilog 단순화."""
    code = strip_comments(code)
    for n in [d[1] for d in re.findall(MODULE_DEF_RE, code)]:
        code = re.sub(rf"\b{re.escape(n)}\b", n + suffix, code)
    return code

In [ ]:
hier = """
module full_adder(input a, input b, input cin, output s, output cout);
  assign {cout, s} = a + b + cin;
endmodule
module top_module(input [1:0] a, input [1:0] b, output [2:0] y);
  wire c;
  full_adder fa0(.a(a[0]), .b(b[0]), .cin(1'b0), .s(y[0]), .cout(c));
  full_adder fa1(.a(a[1]), .b(b[1]), .cin(c), .s(y[1]), .cout(y[2]));
endmodule
"""
print("top 모듈:", auto_top(hier))
print(rename_modules(hier, "_gate"))

### 2-3. 포트 정보 뽑기

원본 데이터에는 Yosys로 미리 뽑은 포트 정보가 들어 있습니다. 여기서는 **원리를 보기 위해** 정규식으로 간단히 흉내 냅니다.
- 포트: 모듈 선언부의 `input [7:0] a` 같은 부분에서 이름과 폭
- 클럭: `always @(posedge clk ...)`에 나오는 `clk` 같은 이름
- 리셋: 이름에 `rst/reset/clr`이 있으면 리셋. 이름이 `_n`으로 끝나거나 `negedge`면 active-low, 민감도 목록(`@(...)`)에 있으면 **비동기**

> ⚠️ 이 정규식 버전은 실습용입니다. Yosys는 제어 흐름을 분석해서 이름과 상관없이 정확히 찾아냅니다. `data/rtl_problems.jsonl`의 포트 정보는 이 함수로 만든 뒤 하나씩 확인했습니다.

In [ ]:
def parse_ports(code, top=None):
    """ANSI 스타일 모듈 헤더에서 (이름, 비트폭) 목록을 뽑는다. CodeV-R1 은 이 일을 Yosys 로 한다."""
    code = strip_comments(code)
    top = top or auto_top(code)
    m = re.search(rf"\bmodule\s+{re.escape(top)}\s*(\#\s*\([\s\S]*?\))?\s*\(([\s\S]*?)\)\s*;", code)
    inputs, outputs, direction, width = [], [], None, 1
    for item in m.group(2).split(","):
        item = item.strip()
        d = re.match(r"(input|output|inout)\b", item)
        if d:
            direction = d.group(1)
            w = re.search(r"\[\s*(\d+)\s*:\s*(\d+)\s*\]", item)
            width = abs(int(w.group(1)) - int(w.group(2))) + 1 if w else 1
        name = re.findall(r"[a-zA-Z_]\w*", re.sub(r"\[[^\]]*\]", "", item))[-1]
        (inputs if direction == "input" else outputs).append([name, width])
    return inputs, outputs


def guess_clock_reset(code, inputs):
    """always @(posedge ...) 에서 클럭/리셋을 추정 (CodeV-R1 은 Yosys 제어 흐름 분석으로 정확히 찾는다)."""
    code = strip_comments(code)
    in_names = {n for n, _ in inputs}
    edges = re.findall(r"\b(posedge|negedge)\s+([a-zA-Z_]\w*)", code)
    async_sigs = {s for _, s in edges}
    clocks, resets = [], []
    for edge, sig in edges:
        if sig in in_names and re.search(r"cl(oc)?k", sig, re.I) and [sig, int(edge == "posedge")] not in clocks:
            clocks.append([sig, int(edge == "posedge")])
    for name, _ in inputs:
        if re.search(r"(rst|reset|clr)", name, re.I):
            active_low = bool(re.search(r"(_n|_b|n)$", name)) or ("negedge", name) in edges
            resets.append([name, 0 if active_low else 1, name not in async_sigs])  # [이름, 활성값, 동기여부]
    return clocks, resets


def port_info_of(code):
    inputs, outputs = parse_ports(code)
    clocks, resets = guess_clock_reset(code, inputs)
    return {"input_port_width": inputs, "output_port_width": outputs,
            "clock_port_polarity": clocks, "reset_port_polarity_sync": resets}

In [ ]:
for pid in ["mux4", "dff_async", "counter4_en", "reg_en_async"]:
    info = port_info_of(by_id[pid]["golden"]["code"])
    same = all(info[k] == by_id[pid]["golden"][k] for k in info)
    print(f"{pid:13} clock={info['clock_port_polarity']}, reset={info['reset_port_polarity_sync']}, 저장된 정보와 일치={same}")

### 2-4. 테스트벤치 자동 생성과 iverilog 실행

원본 `generate_testbench`의 구조를 그대로 따릅니다.

- 입력마다 `reg xxx_in`, 출력마다 `wire xxx_gold`, `wire xxx_gate`
- `trigger = ~(모든 출력이 ===로 같음)`: `===`는 X(모름), Z(끊김)까지 똑같아야 참입니다
- `randomize_inputs`: 클럭, 리셋을 뺀 입력에 `$random`. 32비트보다 넓으면 여러 개를 이어 붙임
- `toggle_clock`: 클럭을 반전 (inner loop 한 번에 반 주기)
- 리셋: (A) 무작위 리셋 값으로 한 바퀴, (B) 리셋 조합마다 "걸고 → 클럭 두 번 → 풀고" 한 바퀴
- 끝나면 `Error rate`를 출력하고, 오류가 0이면 `All tests passed.`

원본과 다른 점: 반복 횟수 기본값을 줄였고(`outer=20, inner=200`; 원본 100 × 1000), 비동기 리셋도 클럭 토글을 기다린 뒤 풀도록 했습니다.

In [ ]:
def generate_testbench(info, gold_top, gate_top, gold_sfx="_gold", gate_sfx="_gate"):
    """CodeV-R1 verify.py 의 generate_testbench 를 줄인 것. gold 와 gate 를 나란히 두고 같은 무작위 입력을 넣는다."""
    ins, outs = info["input_port_width"], info["output_port_width"]
    clocks, resets = info["clock_port_polarity"], info["reset_port_polarity_sync"]
    if len(clocks) > 1:
        raise ValueError("클럭이 여러 개인 회로는 지원하지 않음 (CodeV-R1 도 동일)")
    clk = clocks[0][0] if clocks else None
    rst_names = [r[0] for r in resets]
    L = []
    L += [f"    reg [{w-1}:0] {p}_in;" for p, w in ins]
    L += [f"    wire [{w-1}:0] {p}{gold_sfx}, {p}{gate_sfx};" for p, w in outs]
    L += ["    reg trigger;", "    real num_all = 0, num_errors = 0;",
          "    integer seed, outerLoopNum, innerLoopNum;"]
    conn = lambda sfx: ", ".join([f".{p}({p}_in)" for p, _ in ins] + [f".{p}({p}{sfx})" for p, _ in outs])
    L += [f"    {gold_top}{gold_sfx} gold ({conn(gold_sfx)});",
          f"    {gate_top}{gate_sfx} gate ({conn(gate_sfx)});"]
    # 출력이 하나라도 다르면 trigger = 1  (=== 는 X/Z 까지 똑같아야 참)
    same = " & ".join([f"({p}{gold_sfx} === {p}{gate_sfx})" for p, _ in outs] + ["1'b1"])
    L += [f"    always @(*) begin #5; trigger = ~({same}); end"]
    if clk:
        L += [f"    task toggle_clock; begin {clk}_in = ~{clk}_in; end endtask"]
    rand_lines = " ".join(f"{p}_in = {{{', '.join(['$random(seed)'] * math.ceil(w / 32))}}};"
                          for p, w in ins if p != clk and p not in rst_names)
    L += [f"    task randomize_inputs; begin {rand_lines} end endtask",
          "    task count_errors; begin if (trigger === 1'b1) num_errors = num_errors + 1; "
          "num_all = num_all + 1; end endtask"]
    if resets:
        L += ["    task random_reset; begin " + " ".join(f"{p}_in = $random(seed);" for p in rst_names) + " end endtask"]
        # 리셋 조합마다: 리셋을 걸고 → (클럭이 있으면) 두 번 토글 → 해제
        combos = [list(c) for r in range(1, len(resets) + 1) for c in combinations(resets, r)]
        for i, combo in enumerate(combos):
            on = " ".join(f"{p}_in = {pol};" for p, pol, _ in combo)
            off = " ".join(f"{p}_in = {1 - pol};" for p, pol, _ in combo)
            tog = "#10; toggle_clock; #10; toggle_clock;" if clk else "#10;"
            L += [f"    task reset_{i}; begin {on} {tog} {off} end endtask"]
    else:
        combos = []

    def loop(prefix):
        body = [f"        repeat (outerLoopNum) begin", f"            {prefix}", "            #100; count_errors;",
                "            repeat (innerLoopNum) begin", "                #100; randomize_inputs;"]
        if clk:
            body.append("                #100; toggle_clock;")
        body += ["                #100; count_errors;", "            end", "        end"]
        return body
    L += ["    initial begin",
          '        if (!$value$plusargs("seed=%d", seed)) seed = 0;',
          '        if (!$value$plusargs("outerLoopNum=%d", outerLoopNum)) outerLoopNum = 100;',
          '        if (!$value$plusargs("innerLoopNum=%d", innerLoopNum)) innerLoopNum = 1000;']
    if clk:
        L.append(f"        {clk}_in = {0 if clocks[0][1] else 1};")
    for p, _ in ins:  # 나머지 입력도 X 로 시작하지 않게 0 으로
        if p != clk:
            L.append(f"        {p}_in = 0;")
    L += loop("random_reset;" if resets else "")          # (A) 무작위 리셋 + 무작위 입력
    for i in range(len(combos)):                            # (B) 리셋 조합별로 한 번씩
        L += loop(f"reset_{i};")
    L += ['        $display("Number of all tests:  %0d", num_all);',
          '        $display("Number of errors:     %0d", num_errors);',
          '        $display("Error rate: %.8f", num_errors / num_all);',
          '        if (num_errors == 0) $display("All tests passed.");',
          "        $finish;", "    end"]
    return "module testbench;\n" + "\n".join(L) + "\nendmodule\n"

In [ ]:
g = by_id["counter4_en"]["golden"]
print(generate_testbench(g, "top_module", "top_module"))

생성된 테스트벤치를 golden, gate 코드와 함께 컴파일하고 실행합니다.

원본은 `multiprocessing.Process`로 감싸서 30초가 지나면 프로세스를 죽입니다 (`verify_one_sample_wrapper`). 모델이 쓴 코드가 **무한 루프나 끝나지 않는 시뮬레이션**을 만들 수 있기 때문입니다.
우리는 `subprocess.run(timeout=...)`으로 같은 일을 합니다 (Windows, Jupyter에서도 안전).

In [ ]:
def equiv_check(gold_code, dut_code, info, outer=20, inner=200, seed=0, timeout=20):
    """gold 와 dut 를 같은 테스트벤치에 넣고 iverilog 로 시뮬레이션. CodeV-R1 equiv_with_testbench."""
    try:
        gold_top, gate_top = auto_top(gold_code), auto_top(dut_code)
    except Exception as e:
        return {"correct": False, "error": f"parse: {e}"}
    with tempfile.TemporaryDirectory() as d:
        open(os.path.join(d, "gold.v"), "w").write(rename_modules(gold_code, "_gold"))
        open(os.path.join(d, "gate.v"), "w").write(rename_modules(dut_code, "_gate"))
        open(os.path.join(d, "tb.v"), "w").write(generate_testbench(info, gold_top, gate_top))
        vvp = os.path.join(d, "tb.vvp")
        try:
            c = subprocess.run(["iverilog", "-g2012", "-o", vvp, "-s", "testbench",
                                os.path.join(d, "gold.v"), os.path.join(d, "gate.v"), os.path.join(d, "tb.v")],
                               capture_output=True, text=True, timeout=timeout)
            if c.returncode != 0:
                return {"correct": False, "error": "compile: " + c.stderr.strip()[-500:]}
            s = subprocess.run(["vvp", "-n", vvp, f"+seed={seed}", f"+outerLoopNum={outer}", f"+innerLoopNum={inner}"],
                               capture_output=True, text=True, timeout=timeout)
        except subprocess.TimeoutExpired:
            return {"correct": False, "error": "timeout"}
    m = re.search(r"Error rate:\s*([\d.]+)", s.stdout)
    rate = float(m.group(1)) if m else 1.0
    return {"correct": "All tests passed." in s.stdout, "error_rate": rate}

네 가지 경우를 시험해 봅시다.

In [ ]:
gold = by_id["counter4_en"]["golden"]

cases = {
    "① golden 그대로": gold["code"],
    "② 다르게 썼지만 같은 동작": """
module top_module(input clk, input reset, input en, output reg [3:0] q);
  always @(posedge clk) begin
    if (reset) q <= 4'b0000;
    else if (en) q <= (q == 4'd15) ? 4'd0 : q + 1'b1;
  end
endmodule""",
    "③ 리셋 우선순위 버그": """
module top_module(input clk, input reset, input en, output reg [3:0] q);
  always @(posedge clk) begin
    if (en) q <= q + 4'd1;
    else if (reset) q <= 4'd0;
  end
endmodule""",
    "④ 문법 오류": """
module top_module(input clk, input reset, input en, output [3:0] q);
  always @(posedge clk) q <= q + 1;
endmodule""",
}
for name, dut in cases.items():
    t = time.time()
    r = equiv_check(gold["code"], dut, gold)
    print(f"{name:20} → correct={r['correct']}, error_rate={r.get('error_rate')}, ({time.time() - t:.2f}s) {r.get('error', '')[:80]}")

- ②는 코드가 달라도 **동작이 같으므로 통과**합니다. 문자열 비교로는 불가능한 채점입니다.
- ③은 `en=1`이면서 `reset=1`일 때만 틀리지만, 무작위 입력이 그 상황을 금방 만들어 냅니다.
- ④는 `q`가 `wire`인데 `always`에서 대입해서 컴파일 오류가 납니다 (ASIC 노트 `notes/asic-flow-post-training.md`의 디버그 예와 같은 실수).

### 2-5. 최종 보상 함수: `compute_score`

`codev.py`의 핵심만 옮기면 이것이 전부입니다. **형식이 맞고, 원본과 등가면 1.0, 아니면 0.0.**
원본은 `ground_truth`에 정답 회로가 여러 개 있을 수 있어서 각각과 비교한 뒤 `max`를 취합니다.

In [ ]:
def compute_score(solution_str, golden):
    """CodeV-R1 codev.py 의 compute_score: 형식 + 등가성 → 1.0 아니면 0.0"""
    code = extract_verilog(solution_str)
    if not check_format(solution_str) or code is None:
        return 0.0
    return 1.0 if equiv_check(golden["code"], code, golden)["correct"] else 0.0


def wrap(code, think="ok"):
    return f"<think>{think}</think>  <answer>\n```verilog\n{code}\n```\n</answer>"

ans = gold["code"]
print("정답          :", compute_score(wrap(ans), gold))
print("형식 틀림      :", compute_score(f"<think> <answer></think> ```verilog\n{ans}```</answer>", gold))
print("빈 답          :", compute_score(wrap(""), gold))
print("같은 동작 다른 코드:", compute_score(wrap(cases["② 다르게 썼지만 같은 동작"]), gold))

위 세 줄은 원본 `codev.py`의 `__main__`에 있는 테스트(정답 / 형식 틀림 / 빈 답)와 같습니다.

### 2-6. 이 채점기의 한계: 무작위 테스트는 증명이 아니다

시뮬레이션 기반 검사는 **"수많은 입력에서 같았다"** 일 뿐, 모든 입력에서 같다는 증명은 아닙니다.
예를 들어 32비트 비교기에 아주 드문 입력에서만 틀리는 버그(백도어)가 있으면 어떨까요?

In [ ]:
eq_gold = """module top_module(input [31:0] a, input [31:0] b, output eq);
  assign eq = (a == b);
endmodule"""
eq_backdoor = """module top_module(input [31:0] a, input [31:0] b, output eq);
  assign eq = (a == b) | (a == 32'hDEADBEEF);   // a가 이 값일 때만 틀림
endmodule"""
info = port_info_of(eq_gold)
print("백도어 비교기:", equiv_check(eq_gold, eq_backdoor, info))

무작위 입력 4,000개 중에 `a == 32'hDEADBEEF`가 나올 확률은 거의 0이라서 **틀린 코드가 통과합니다 (false positive)**.
반대로 golden이 X(미정) 값을 내는 동안의 차이 때문에 맞는 코드가 실패하는 경우도 있습니다 (false negative).

논문이 보고한 수치는 이렇습니다.
- golden 코드끼리 비교했을 때 실패로 잘못 판정하는 비율: **0.3%** (LLM이 쓴 테스트벤치는 7.6%)
- DeepSeek-V3가 순차회로에 일부러 심은 미묘한 버그를 잡아내는 비율: **65%** (LLM 테스트벤치는 40%)

즉 완벽하지 않지만, **LLM에게 테스트벤치를 쓰게 하는 것보다 훨씬 정확하고, 사람 없이 대량으로 돌릴 수 있다**는 것이 핵심입니다.
더 확실하게 하려면 Yosys/SymbiYosys의 형식 등가성 검사(formal equivalence)를 쓸 수 있지만 느리고 모든 회로에 적용하기 어렵습니다.

> 💡 RL 모델은 채점기의 허점을 파고듭니다(reward hacking, 5단계 6절). 보상이 "테스트벤치 통과"라면,
> 테스트벤치가 놓치는 부분을 대충 쓰는 모델이 될 수 있습니다. 실제 서비스용이라면 반복 횟수를 늘리고, 방향성 테스트(directed test)나 형식 검증을 섞습니다.

## 3. 데이터 만들기: round-trip 합성과 필터링

### 3-1. CodeV-R1의 데이터 파이프라인

좋은 (스펙, 코드) 쌍은 구하기 어렵습니다. 사람이 스펙을 쓰기도 비싸고, LLM이 쓴 코드는 틀릴 수 있습니다. CodeV-R1의 해법은 **왕복(round-trip)** 입니다.

| 단계 | 하는 일 | 왜 |
|---|---|---|
| ① code → NL | GitHub의 **진짜** Verilog 코드를 DeepSeek-V3에게 주고 스펙을 쓰게 함 (약 150K) | 코드는 진짜라 정답이 보장됨 |
| ② NL → code | DeepSeek-R1이 스펙만 보고 **생각 + 코드** 작성 | SFT로 배울 "생각하는 법" |
| ③ 등가성 검사 | ②의 코드가 ①의 원본 코드와 등가인지 2절의 방법으로 검사 | 스펙이 원본을 제대로 설명했고, R1의 풀이가 맞다는 증거 |
| ④ 난이도 필터 | Qwen2.5-Coder-7B/32B-Instruct가 5번 중 한 번이라도 맞히는 문제는 제거 | 쉬운 문제는 배울 게 없음 |
| ⑤ 오염 제거 | 벤치마크(VerilogEval, RTLLM)와 Rouge-L > 0.5인 문제 제거 | 시험 문제 유출 방지 |
| ⑥ 합성 가능성 | Yosys로 합성되는 golden 코드만 남김 | 실제 하드웨어가 되는 코드만 |

결과가 SFT용 약 **87K**입니다. ③은 사실 **"교사 모델의 답 중 맞은 것만 남기는" rejection sampling** 입니다.
스펙이 엉터리여도, R1의 답이 틀려도 걸러지므로 데이터 품질이 자동으로 관리됩니다.

### 3-2. 직접 해 보기: 교사 응답 거르기

우리 데이터에서는 `rtl_problems.jsonl`의 `golden`이 ①의 "GitHub 원본 코드", `rtl_teacher_responses.jsonl`이 ②의 "R1 응답" 역할입니다.
교사 응답에는 일부러 틀린 것(OR을 AND로, 리셋 우선순위, 회전 대신 시프트, 블로킹 대입 등)이 섞여 있습니다. 2절의 채점기로 걸러 봅시다.

In [ ]:
def score_many(texts, goldens, workers=8):
    """여러 답안을 병렬로 채점 (원본 verl은 reward_manager=prime 으로 병렬 채점)"""
    with ThreadPoolExecutor(max_workers=workers) as ex:
        return list(ex.map(compute_score, texts, goldens))

t = time.time()
scores = score_many([r["response"] for r in teacher_responses],
                    [by_id[r["id"]]["golden"] for r in teacher_responses])
print(f"채점 {len(scores)}개, {time.time() - t:.1f}초")

kept = [r for r, s in zip(teacher_responses, scores) if s == 1.0]
dropped = [r for r, s in zip(teacher_responses, scores) if s == 0.0]
print(f"통과 {len(kept)}개 / 탈락 {len(dropped)}개")
print("\n탈락한 응답의 생각 과정 (틀린 이유가 보이나요?):")
for r in dropped:
    think = r["response"].split("</think>")[0].replace("<think>", "").strip()
    print(f"  - {r['id']:12}: {think}")

틀린 응답의 "생각"을 보면 그럴듯하지만 스펙을 잘못 이해했습니다. 이런 답으로 SFT를 하면 **틀린 추론을 그대로 배웁니다**.
그래서 증류 데이터에는 반드시 검증 필터가 필요합니다.

통과한 응답으로 SFT 데이터를 만듭니다. TRL의 **대화형 prompt / completion** 형식을 씁니다.
`prompt`는 (system, user) 메시지, `completion`은 assistant 메시지입니다. TRL이 chat template을 입히고 **completion 부분만 loss**를 계산합니다 (5단계와 같은 `-100` 마스킹).

In [ ]:
from datasets import Dataset

sft_rows = [{"prompt": build_messages(by_id[r["id"]]),
             "completion": [{"role": "assistant", "content": r["response"]}]} for r in kept]
random.Random(0).shuffle(sft_rows)
sft_dataset = Dataset.from_list(sft_rows)
print(sft_dataset)
print("\n[completion 예시]\n" + sft_dataset[0]["completion"][0]["content"])

## 4. Stage 1: 증류 SFT

### 4-1. 원본 설정 읽기 (`sft/codev_r1_distill.yaml`)

```yaml
model_name_or_path: Qwen/Qwen2.5-Coder-7B-Instruct
stage: sft
finetuning_type: full                                    # LoRA 아님, 7B 전체 학습
deepspeed: examples/deepspeed/ds_z3_offload_config.json  # ZeRO-3 + CPU offload로 메모리 분산
dataset: codev_r1_sft
template: qwen
cutoff_len: 16384                                        # R1의 긴 생각 과정을 자르지 않도록
per_device_train_batch_size: 1
gradient_accumulation_steps: 8                           # GPU 8장 × 1 × 8 = 실효 배치 64
learning_rate: 1.0e-5
num_train_epochs: 6.0
lr_scheduler_type: cosine
warmup_ratio: 0.03
bf16: true
packing: true                                            # 짧은 예제들을 한 시퀀스에 이어 붙여 낭비 줄임
flash_attn: fa2
```

| LLaMA-Factory | TRL `SFTConfig` | 원본 | 이번 실습 | 왜 다르게 |
|---|---|---|---|---|
| `finetuning_type: full` | `peft_config` 없음 | Full | **LoRA** | 3단계: 12GB에서 안정적 |
| `cutoff_len` | `max_length` | 16384 | 1024 (CPU 768) | 교사 응답이 짧음 |
| `learning_rate` | `learning_rate` | 1e-5 | 2e-4 | LoRA는 학습률을 크게 (3단계) |
| `num_train_epochs` | `num_train_epochs` | 6 | 4 (CPU 3) | 데이터가 작음 |
| `packing` | `packing` | true | false | 데이터가 작아 효과 없음 |
| `template: qwen` | 토크나이저의 chat template | qwen | 같음 | |

> 💡 학습률 1e-5, 6 epoch는 87K개 × 평균 수천 토큰이라는 큰 데이터에서의 값입니다. 이 단계에서 7B 모델이 **R1의 말투와 추론 습관**을 배웁니다.
> 논문의 CodeV-R1-7B-Distill은 이것만으로 VerilogEval v2 spec-to-RTL pass@1 65.2%를 냅니다 (베이스 Qwen2.5-Coder-7B-Instruct보다 크게 향상).

### 4-2. 모델 불러오기와 평가 함수

평가는 원본 `test/`와 같은 방식입니다. 문제마다 답을 여러 개 뽑고(temperature 0.6), 채점해서 **pass@1 = 맞은 비율의 평균**을 냅니다.
세 가지 숫자를 같이 봅니다.
- **형식 OK:** `<think>…</think><answer>…</answer>` 형식을 지켰는가
- **기능 통과:** 형식과 상관없이, 마지막 ```verilog 블록이 golden과 등가인가 (VerilogEval식 채점)
- **보상(엄격):** 형식 OK **그리고** 기능 통과 = RL에서 받는 보상

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.padding_side = "left"          # 생성은 왼쪽 패딩
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

dtype = torch.bfloat16 if config["bf16"] else (torch.float16 if config["fp16"] else torch.float32)
base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=dtype).to(device)
print(f"파라미터 수: {sum(p.numel() for p in base_model.parameters()) / 1e6:.0f}M, dtype={dtype}")

In [ ]:
@torch.no_grad()
def generate(model, problem_list, n, temperature, max_new_tokens=None):
    """문제마다 n개씩 답을 생성한다. 반환: [[답1, 답2, ...], ...]"""
    model.eval()
    max_new_tokens = max_new_tokens or config["gen_max_new_tokens"]
    results = []
    step = max(1, config["gen_batch"] // n)
    for i in range(0, len(problem_list), step):
        chunk = problem_list[i:i + step]
        texts = [tokenizer.apply_chat_template(build_messages(p), tokenize=False, add_generation_prompt=True)
                 for p in chunk]
        enc = tokenizer(texts, return_tensors="pt", padding=True).to(model.device)
        out = model.generate(**enc, do_sample=True, temperature=temperature, top_p=0.95,
                             num_return_sequences=n, max_new_tokens=max_new_tokens,
                             pad_token_id=tokenizer.pad_token_id)
        dec = tokenizer.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
        results += [dec[j * n:(j + 1) * n] for j in range(len(chunk))]
    return results


def functional_pass(text, golden):
    code = extract_verilog(text)
    return code is not None and equiv_check(golden["code"], code, golden)["correct"]


def evaluate(model, name, problem_list=None, n=None):
    problem_list = problem_list or splits["eval"]
    n = n or config["eval_samples"]
    t = time.time()
    outs = generate(model, problem_list, n, config["eval_temperature"])
    flat = [(o, p["golden"]) for p, os_ in zip(problem_list, outs) for o in os_]
    with ThreadPoolExecutor(8) as ex:
        func = list(ex.map(lambda x: functional_pass(*x), flat))
    fmt = [check_format(o) for o, _ in flat]
    strict = [f and c for f, c in zip(fmt, func)]
    report = {"형식 OK": sum(fmt) / len(fmt), "기능 통과": sum(func) / len(func), "보상(엄격)": sum(strict) / len(strict)}
    print(f"[{name}] " + ", ".join(f"{k}: {v:.2f}" for k, v in report.items()) + f"  ({time.time() - t:.0f}초)")
    per_problem = [sum(strict[i * n:(i + 1) * n]) for i in range(len(problem_list))]
    return outs, report, per_problem

all_reports = {}
base_outs, all_reports["base"], _ = evaluate(base_model, "base")
print("\n[base 모델 답 예시]\n" + base_outs[0][0][:800])

베이스 Instruct 모델은 Verilog를 어느 정도 쓰지만, **`<think>/<answer>` 형식을 모르기 때문에 엄격한 보상은 거의 0**입니다.
이 상태로 RL을 시작하면 그룹 안의 답이 전부 0점이라 advantage가 0이 되어 **아무것도 배우지 못합니다** (5단계 5절).
SFT가 RL의 "출발점(cold start)"을 만들어 주는 이유가 이것입니다.

### 4-3. TRL로 SFT 하기

In [ ]:
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

def make_lora_config():
    return LoraConfig(
        r=16, lora_alpha=32, lora_dropout=0.05,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        task_type="CAUSAL_LM",
    )

sft_args = SFTConfig(
    output_dir=str(OUTPUT_DIR / "sft"),
    num_train_epochs=config["sft_epochs"],
    per_device_train_batch_size=config["sft_batch_size"],
    learning_rate=config["sft_lr"],
    lr_scheduler_type="cosine",
    warmup_steps=2,
    max_length=config["sft_max_length"],
    completion_only_loss=True,          # assistant 답변만 loss
    bf16=config["bf16"],
    fp16=config["fp16"],
    logging_steps=2,
    save_strategy="no",
    report_to="none",
    seed=42,
)
sft_trainer = SFTTrainer(
    model=base_model,
    args=sft_args,
    train_dataset=sft_dataset,
    processing_class=tokenizer,
    peft_config=make_lora_config(),
)
sft_trainer.model.print_trainable_parameters()

batch = sft_trainer.data_collator([sft_trainer.train_dataset[0]])
ids, labels = batch["input_ids"][0], batch["labels"][0]
print("\nloss 계산 부분 (앞 300자):", repr(tokenizer.decode(ids[labels != -100])[:300]))

In [ ]:
if device == "cuda":
    torch.cuda.reset_peak_memory_stats()
start = time.time()
sft_trainer.train()
print(f"\nSFT 학습 시간: {(time.time() - start) / 60:.1f}분, 최대 VRAM: {peak_vram():.2f} GB")

sft_model = sft_trainer.model.merge_and_unload()     # LoRA를 합쳐서 RL의 출발점으로
del sft_trainer, base_model
free_memory()

sft_outs, all_reports["sft"], _ = evaluate(sft_model, "SFT")
print("\n[SFT 모델 답 예시]\n" + sft_outs[0][0][:800])

SFT 뒤에는 **형식 OK가 크게 오릅니다**. 교사의 "생각 → 답" 습관을 따라 하게 된 것입니다.
기능 통과는 데이터가 20개 정도라서 조금 오르거나 비슷할 수 있습니다. 원본은 87K개로 이 단계에서 정답률이 크게 오릅니다.

> 💡 0.5B 모델과 20개 데이터라서 결과가 들쭉날쭉합니다. 평가 문제도 8개뿐이라 한 문제가 12.5%p입니다. 숫자보다 **흐름**을 보세요.

## 5. RL 데이터 고르기: 난이도 필터

GRPO/DAPO는 **그룹 안에 맞은 답과 틀린 답이 섞여 있어야** 학습 신호가 생깁니다 (advantage = 보상 − 그룹 평균).
- 모델이 항상 맞히는 문제 → 모두 1점 → advantage 0 → 계산만 낭비
- 모델이 절대 못 맞히는 문제 → 모두 0점 → advantage 0

CodeV-R1은 RL 데이터 3.1K를 이렇게 골랐습니다.
1. R1의 답이 golden과 **등가인 문제** (풀 수 있고, 스펙에 오류가 없는 문제)
2. Qwen2.5-Coder-7B/32B-Instruct가 5번 다 **틀리는** 문제 (어려운 문제)
3. SFT 모델(CodeV-R1-Distill)로 5번 뽑아서 **5번 다 맞히는 문제는 제외**

> 논문의 ablation: 난이도 필터 없이 16K 데이터를 쓰거나, 등가성 검사 없이 R1 답을 정답으로 쓴 14K 데이터를 썼을 때보다 **3.1K가 보상도 높고 학습도 안정적**이었습니다. RL 데이터는 양보다 질입니다.

우리도 3번 규칙을 SFT 모델로 적용해 봅니다.

In [ ]:
rl_candidates = splits["rl"]
k = config["filter_samples"]
t = time.time()
cand_outs = generate(sft_model, rl_candidates, n=k, temperature=1.0)
cand_scores = [score_many(o, [p["golden"]] * len(o)) for p, o in zip(rl_candidates, cand_outs)]
print(f"샘플링 + 채점: {time.time() - t:.0f}초\n")

rl_problems = []
for p, s in zip(rl_candidates, cand_scores):
    n_ok = int(sum(s))
    keep = n_ok < k                    # 전부 맞힌 문제만 제외 (CodeV-R1 규칙)
    print(f"{p['id']:14} {n_ok}/{k} 맞음 → {'사용' if keep else '제외 (너무 쉬움)'}")
    if keep:
        rl_problems.append(p)
print(f"\nRL에 쓸 문제: {len(rl_problems)}개")

`0/k`인 문제도 남깁니다. 지금은 못 풀어도, RL 중에 temperature 1.0으로 여러 번 뽑다 보면 가끔 맞히는 답이 나오고, 그때부터 학습 신호가 생깁니다.
그 대신 학습 중에 "전부 맞음/전부 틀림" 그룹을 걸러 내는 것이 DAPO의 **dynamic sampling** 입니다 (다음 절).

## 6. Stage 2: RL (DAPO)

### 6-1. GRPO에서 DAPO로

DAPO(Decoupled clip and Dynamic sAmpling Policy Optimization)는 GRPO를 **긴 추론(long CoT) RL**에 맞게 고친 레시피입니다. 5단계의 GRPO와 달라진 점은 네 가지입니다.

| 기법 | GRPO (5단계) | DAPO | verl 설정 (`run_dapo_codev_7b_3.1k.sh`) | TRL `GRPOConfig` |
|---|---|---|---|---|
| **Clip-higher** | 확률 비율을 [1−ε, 1+ε]로 자름 | 위쪽만 넓게: [1−0.2, 1+**0.28**]. 확률이 낮던 좋은 토큰이 더 빨리 올라가서 **탐색(entropy)이 유지됨** | `clip_ratio_low=0.2`, `clip_ratio_high=0.28` | `epsilon=0.2`, `epsilon_high=0.28` |
| **Token-level loss** | 답안별 평균 후 평균 | 모든 토큰을 한꺼번에 평균. 긴 답안의 토큰이 손해 보지 않음 | `use_token_level_loss=True` | `loss_type="dapo"` |
| **Dynamic sampling** | 없음 | 그룹 전체가 0점이나 1점이면 버리고 **더 뽑아서 배치를 채움** | `filter_groups.enable=True`, `metric=acc` | (없음, 5절 오프라인 필터로 대신) |
| **Overlong reward shaping** | 없음 | 최대 길이에 가까워지면 점점 감점 | `overlong_buffer.len=1024`, `penalty_factor=1.0` | `trl.rewards.get_soft_overlong_punishment` |
| **KL 항** | β로 기준 모델과의 거리 제한 | **없음** (긴 추론에서는 기준 모델에서 많이 멀어져야 함) | `kl_coef=0.0`, `use_kl_loss=False` | `beta=0.0` |

> 💡 KL을 없애도 괜찮은 이유: 보상이 **검증 가능한 정답**이라 reward model을 속이는 식의 hacking이 어렵고, 모델이 SFT 모델에서 멀어지는 것이 곧 실력 향상이기 때문입니다.

### 6-2. CodeV-R1의 adaptive DAPO

DAPO의 dynamic sampling에는 비용 문제가 있습니다. 배치 128개를 채우려고 생성했는데 절반이 "전부 맞음/전부 틀림"이면, **한 번 더 생성**해야 합니다. 생성은 RL에서 가장 비싼 부분입니다.
CodeV-R1은 **지난 스텝의 유효 비율을 보고 처음부터 넉넉하게 뽑는** 방식으로 바꿨습니다.

$$ b_{gen} = \left\lceil \frac{b_{train}}{r_{valid}} \right\rceil, \qquad r_{valid} \leftarrow \min\left(r_{valid},\ \frac{\text{유효 그룹 수}}{b_{gen}}\right) $$

논문에 따르면 전체 학습이 **1.25배** 빨라졌고, 모델이 쉬운 문제를 다 맞히게 되는 학습 후반(스텝 150 이후)에는 1.44배였습니다.
간단한 시뮬레이션으로 차이를 확인해 봅시다. (실제 모델 없이, 문제마다 "맞힐 확률"이 있다고 가정)

In [ ]:
def simulate(adaptive, steps=60, b_train=128, G=16, seed=0):
    rng = random.Random(seed)
    skill = 0.0                                   # 학습이 진행될수록 문제를 더 잘 맞힘
    r_valid, rounds, sampled = 1.0, [], 0
    for step in range(steps):
        skill = step / steps
        valid, n_round = 0, 0
        while valid < b_train:                    # 배치가 찰 때까지 생성 반복
            b_gen = math.ceil(b_train / r_valid) if adaptive else b_train
            n_round += 1
            sampled += b_gen
            got = 0
            for _ in range(b_gen):
                p = min(1.0, max(0.0, rng.random() * 0.7 + skill * 0.6 - 0.1))   # 이 문제의 정답 확률
                correct = sum(rng.random() < p for _ in range(G))
                got += 0 < correct < G            # 맞음/틀림이 섞인 그룹만 유효
            valid += got
            if adaptive:
                r_valid = min(r_valid, got / b_gen) if got else r_valid / 2
        rounds.append(n_round)
    return rounds, sampled

for name, adaptive in [("DAPO (고정 b_gen)", False), ("adaptive DAPO", True)]:
    rounds, sampled = simulate(adaptive)
    print(f"{name:18}: 스텝당 생성 횟수 평균 {sum(rounds) / len(rounds):.2f}, 후반 30스텝 {sum(rounds[30:]) / 30:.2f}, 생성한 프롬프트 수 {sampled}")

고정 방식은 학습 후반에 유효 그룹이 줄어들면서 **한 스텝에 생성을 여러 번** 해야 합니다. 생성은 순서대로 기다려야 하므로 그만큼 느립니다.
adaptive 방식은 한 번에 넉넉히 뽑아서 대부분 **한 번에** 끝냅니다. (조금 더 많이 뽑지만 병렬로 처리되므로 이득)

TRL `GRPOTrainer`에는 dynamic sampling이 없으므로, 이번 실습은 5절의 오프라인 필터로 대신합니다.

### 6-3. 원본 RL 설정 읽기

`verl/recipe/dapo/run_dapo_codev_7b_3.1k.sh`의 핵심 줄과 TRL 대응입니다.

| verl 설정 | 원본 값 | 의미 | TRL `GRPOConfig` | 이번 실습 (GPU) |
|---|---|---|---|---|
| `algorithm.adv_estimator` | `grpo` | 그룹 상대 advantage | (기본) | 같음 |
| `data.train_batch_size` | 128 | 스텝당 프롬프트 수 | `per_device_train_batch_size × grad_accum / num_generations` | 2 |
| `actor_rollout_ref.rollout.n` | 16 | 그룹 크기 G | `num_generations` | 8 |
| `data.max_response_length` | 16384 | 생성 최대 길이 | `max_completion_length` | 512 |
| `actor.optim.lr` | 1e-6 | 학습률 (Full) | `learning_rate` | 2e-5 (LoRA) |
| `rollout.temperature` | 1.0 | 탐색을 위해 높게 | `temperature` | 1.0 |
| `actor.clip_ratio_low/high` | 0.2 / 0.28 | clip-higher | `epsilon` / `epsilon_high` | 같음 |
| `use_kl_loss`, `kl_coef` | False, 0.0 | KL 없음 | `beta` | 0.0 |
| `actor.grad_clip` | 0.5 | gradient clipping | `max_grad_norm` | 0.5 |
| `custom_reward_function.train.path` | `codev.py` | 보상 함수 | `reward_funcs` | 2-5절 `compute_score` |
| `rollout.name` | `vllm` | 빠른 생성 엔진 | `use_vllm` | False (HF generate) |
| `trainer.total_epochs` | 100 (논문: 약 300스텝) | | `max_steps` | 40 |

### 6-4. TRL용 보상 함수 만들기

TRL의 보상 함수는 `(prompts, completions, **데이터셋의 다른 칸)`을 받아 점수 리스트를 돌려줍니다.
`golden`은 데이터셋에 **JSON 문자열**로 넣습니다. 포트 정보가 `["rst_n", 0, false]`처럼 타입이 섞인 리스트라 Arrow 표에 그대로 못 넣기 때문입니다 (원본이 `pickle` 바이트로 넣는 것과 같은 이유).

In [ ]:
from trl.rewards import get_soft_overlong_punishment

def codev_reward(completions, golden, **kwargs):
    texts = [c[0]["content"] if isinstance(c, list) else c for c in completions]
    goldens = [json.loads(g) for g in golden]
    return score_many(texts, goldens)

# DAPO의 overlong reward shaping: 마지막 1/4 구간에 들어가면 점점 감점, 넘치면 -1
overlong_penalty = get_soft_overlong_punishment(
    max_completion_len=config["grpo_max_completion"],
    soft_punish_cache=config["grpo_max_completion"] // 4,
)
reward_funcs = [codev_reward, overlong_penalty]

grpo_dataset = Dataset.from_list([{"prompt": build_messages(p), "golden": json.dumps(p["golden"])}
                                  for p in rl_problems])
print(grpo_dataset)

# 동작 확인: 교사 정답 하나를 넣어 보기
p0 = by_id["counter4_en"]
print("보상 확인:", codev_reward([[{"role": "assistant", "content": wrap(p0["golden"]["code"])}]],
                              [json.dumps(p0["golden"])]))

### 6-5. 학습

5단계 GRPO 설정에서 DAPO에 해당하는 값만 바꿨습니다. `log_completions=True`로 두면 학습 중 생성된 답을 볼 수 있습니다.

In [ ]:
from trl import GRPOConfig, GRPOTrainer

grpo_args = GRPOConfig(
    output_dir=str(OUTPUT_DIR / "rl"),
    max_steps=config["grpo_steps"],
    per_device_train_batch_size=config["grpo_batch_size"],
    gradient_accumulation_steps=config["grpo_grad_accum"],
    num_generations=config["grpo_num_generations"],
    max_completion_length=config["grpo_max_completion"],
    temperature=1.0,
    learning_rate=config["grpo_lr"],
    # ---- DAPO ----
    loss_type="dapo",            # token-level loss
    epsilon=0.2,
    epsilon_high=0.28,           # clip-higher
    beta=0.0,                    # KL 없음
    mask_truncated_completions=False,   # 논문: DAPO의 overlong filtering은 쓰지 않음 (대신 overlong 감점)
    max_grad_norm=0.5,
    # ----------------
    bf16=config["bf16"],
    fp16=config["fp16"],
    logging_steps=1,
    save_strategy="no",
    report_to="none",
    seed=42,
)
grpo_trainer = GRPOTrainer(
    model=sft_model,
    reward_funcs=reward_funcs,
    args=grpo_args,
    train_dataset=grpo_dataset,
    processing_class=tokenizer,
    peft_config=make_lora_config(),
)

if device == "cuda":
    torch.cuda.reset_peak_memory_stats()
start = time.time()
grpo_trainer.train()
print(f"\nRL 학습 시간: {(time.time() - start) / 60:.1f}분, 최대 VRAM: {peak_vram():.2f} GB")

### 6-6. 학습 곡선 보기

- **codev 보상 평균:** 오르면 정답률이 오르는 중입니다
- **답 길이:** CodeV-R1 논문에서는 RL 중에 생각 과정이 점점 길어집니다 (DeepSeek-R1과 같은 현상). 우리처럼 짧은 학습에서는 잘 안 보일 수 있습니다
- **`frac_reward_zero_std`:** 그룹 안의 답이 **전부 같은 점수**(모두 0점 또는 모두 1점)인 비율입니다. 이 그룹들은 advantage가 0이라 학습에 기여하지 않고, DAPO의 dynamic sampling이라면 버리고 다시 뽑았을 그룹입니다. 이 값이 1에 가까우면 문제가 너무 쉽거나 너무 어렵다는 뜻입니다

In [ ]:
logs = [h for h in grpo_trainer.state.log_history if "reward" in h]
steps = [h["step"] for h in logs]
key_reward = next((k for k in logs[0] if k.startswith("rewards/codev_reward") and k.endswith("/mean")), "reward")
key_len = next((k for k in logs[0] if k in ("completions/mean_length", "completion_length")), None)

fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
axes[0].plot(steps, [h[key_reward] for h in logs], marker="."); axes[0].set_title("codev 보상 (그룹 평균)")
if key_len:
    axes[1].plot(steps, [h[key_len] for h in logs], marker=".", color="tab:orange"); axes[1].set_title("답 길이 (토큰)")
axes[2].plot(steps, [h.get("frac_reward_zero_std", float("nan")) for h in logs], marker=".", color="tab:green"); axes[2].set_title("점수가 모두 같은 그룹 비율")
for ax in axes:
    ax.set_xlabel("step"); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()
print("사용한 로그 키:", key_reward, "/", key_len)

## 7. 결과 비교와 pass@k

RL 모델을 합치고 평가 세트로 비교합니다.

In [ ]:
rl_model = grpo_trainer.model.merge_and_unload()
del grpo_trainer, sft_model
free_memory()

rl_outs, all_reports["rl"], rl_per_problem = evaluate(rl_model, "RL")

print(f"\n{'':6}" + "".join(f"{k:>12}" for k in all_reports["base"]))
for name, rep in all_reports.items():
    print(f"{name:6}" + "".join(f"{v:12.2f}" for v in rep.values()))

### pass@k는 어떻게 계산하나

VerilogEval, RTLLM, CodeV-R1 논문은 문제마다 n개(논문은 20개)를 뽑고, 그중 c개가 맞았을 때 **"k번 시도해서 한 번이라도 맞을 확률"** 을 이렇게 추정합니다 (HumanEval의 unbiased estimator).

$$ \text{pass@}k = 1 - \frac{\binom{n-c}{k}}{\binom{n}{k}} $$

k=1이면 그냥 c/n입니다. 위 표의 "보상(엄격)"이 pass@1입니다.

In [ ]:
def pass_at_k(n, c, k):
    if n - c < k:
        return 1.0
    return 1.0 - math.comb(n - c, k) / math.comb(n, k)

n = config["eval_samples"]
for k in [1, n]:
    score = sum(pass_at_k(n, c, k) for c in rl_per_problem) / len(rl_per_problem)
    print(f"RL 모델 pass@{k}: {score:.2f}")
for p, c in zip(splits["eval"], rl_per_problem):
    print(f"  {p['id']:15} {c}/{n}")

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
rl_model.save_pretrained(OUTPUT_DIR / "final")
tokenizer.save_pretrained(OUTPUT_DIR / "final")
print("저장:", OUTPUT_DIR / "final")

print("\n[RL 모델 답 예시]\n" + rl_outs[0][0])

**결과 읽는 법.** 0.5B 모델, 문제 수십 개, 수십 스텝이라 원본처럼 큰 향상을 기대하긴 어렵습니다. 대신 이런 것을 확인하세요.
- base → SFT: **형식 OK가 크게 오른다** (증류가 형식과 추론 습관을 가르침)
- SFT → RL: 학습 문제의 **보상 평균이 오른다**. 평가 세트 숫자는 문제가 8개라 오르내릴 수 있음
- 원본 논문: Distill → RL로 VerilogEval v2 spec-to-RTL 65.2% → **68.8%**, RTLLM v1.1 57.4% → **72.9%** (7B가 671B DeepSeek-R1의 64.8%를 넘음)

> 💡 더 해 보기: `config`에서 `grpo_steps`를 늘리거나, `MODEL_NAME`을 `Qwen/Qwen2.5-Coder-1.5B-Instruct`로 바꿔 보세요 (12GB에서 LoRA로 가능, 3단계 참고).

## 8. 실제 규모로 돌리려면

원본을 그대로 재현하려면 이 순서입니다. (A100 80GB 기준, RTX 3060에서는 불가능)

**① 데이터 받기**
```bash
huggingface-cli download zhuyaoyu/CodeV-R1-dataset --repo-type dataset --local-dir data/codev
# codev_r1_sft.jsonl, codev_r1_rl_train.parquet, codev_r1_rl_val.parquet
```

**② SFT (8 × A100, 약 78시간)**
```bash
git clone https://github.com/hiyouga/LLaMA-Factory && cd LLaMA-Factory && pip install -e .
cp ../CodeV-R1/sft/codev_r1_distill.yaml examples/train_full/
# data/dataset_info.json 에 1-1절의 "codev_r1_sft" 항목 추가
llamafactory-cli train examples/train_full/codev_r1_distill.yaml
```

**③ RL (16 × A100, 약 127시간)**
```bash
cd CodeV-R1/verl
conda create -n verl-codev-r1 python=3.10 conda-forge::iverilog && conda activate verl-codev-r1
pip install torch==2.6.0 && pip install -r requirements.txt && pip install -e .
# recipe/dapo/run_dapo_codev_7b_3.1k.sh 의 MODEL_PATH(SFT 결과), DATA_PATH 수정
# train.parquet, val.parquet 이름으로 RL 데이터 배치
bash run-direct.sh        # 또는 sbatch train-multigpu.slurm
```

**④ 평가**
```bash
cd CodeV-R1/test
bash scripts/preprocess_test.sh
bash benchmark_runner.sh --benchmark verilogeval_v2 --model_path $MODEL_PATH --model_name codev-r1
```

**RTX 3060 12GB에서 현실적인 다음 단계**

| 하고 싶은 것 | 방법 |
|---|---|
| 원본 데이터 일부로 SFT | `codev_r1_sft.jsonl`에서 응답이 짧은(예: 2K 토큰 이하) 예제 수천 개를 골라 1.5B + LoRA(또는 4단계의 QLoRA) |
| 원본 RL 데이터 사용 | parquet의 `reward_model.ground_truth`를 `pickle.loads`하면 우리 `golden`과 같은 구조 (`["answer"]` 안). `set`을 `list`로 바꿔 JSON으로 저장하면 이 노트북의 `codev_reward`를 그대로 쓸 수 있음 |
| 이미 학습된 모델 써 보기 | `zhuyaoyu/CodeV-R1-Qwen-7B`를 4bit로 불러 추론 (생각 과정이 길어 `max_new_tokens`를 4K 이상으로) |
| 보상을 더 엄격하게 | `equiv_check`의 `outer/inner`를 원본 값(100 × 1000)으로, 또는 Yosys `equiv_*`로 형식 등가성 검사 추가 |

> ⚠️ 원본 테스트벤치 생성기가 지원하지 않는 것: 클럭이 여러 개인 회로, 양쪽 엣지를 쓰는 회로, `inout` 포트. 이런 문제는 RL 데이터에서 빠져 있습니다.

## 9. 정리

| 개념 | 한 줄 요약 |
|---|---|
| 증류 SFT | 큰 추론 모델(R1)의 풀이를 따라 쓰게 해서 **형식과 추론 습관**을 옮겨 심음. RL의 출발점(cold start) |
| round-trip 데이터 | 진짜 코드 → LLM이 스펙 작성 → LLM이 다시 코드 작성 → **원본과 등가인 것만** 남김 |
| 등가성 보상 | golden과 모델 코드에 같은 무작위 입력을 넣고 출력 비교. 문자열이 달라도 동작이 같으면 정답 |
| 채점기의 한계 | 무작위 테스트는 증명이 아님 (드문 입력의 버그를 놓침). RL은 그 틈을 파고들 수 있음 |
| 난이도 필터 | 항상 맞히거나 절대 못 맞히는 문제는 학습 신호 0. 적당히 어려운 문제만 RL에 |
| DAPO | clip-higher, token-level loss, dynamic sampling, overlong 감점, KL 없음 |
| adaptive DAPO | 지난 스텝의 유효 비율로 생성량을 미리 늘려서 재생성을 줄임 (1.25배 빠름) |

**연습 문제**
1. `data/rtl_problems.jsonl`에 새 문제(예: 8비트 시프트-가산 곱셈기)를 추가하고, `port_info_of`로 포트 정보를 만든 뒤 golden끼리 `equiv_check`가 통과하는지 확인해 보세요.
2. 2-6절의 백도어 비교기를 잡아내도록 `generate_testbench`에 **방향성 테스트**(예: 한 입력을 다른 입력과 같게 만들기, 0/최댓값 넣기)를 추가해 보세요. 원본 코드에도 `directed_tests` 자리가 `[TODO]`로 남아 있습니다.
3. `codev_reward`에 "컴파일은 되지만 틀림"이면 0.1점을 주는 부분 점수를 넣고 RL 결과를 비교해 보세요. ASIC 노트(`notes/asic-flow-post-training.md`)에서 말한 "컴파일 보상을 크게 주면 빈 모듈로 수렴" 현상이 생기는지 관찰해 보세요.